# Lab 10 — Solución
Solución compacta de los ejercicios: clasificación con `make_moons` y `load_digits` usando Keras.

In [ ]:
# Imports y configuración común
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import tensorflow as tf
from tensorflow import keras

seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)

## Ejercicio 1 — `make_moons` (clasificación binaria)

In [ ]:
# Datos
X, y = make_moons(n_samples=1000, noise=0.15, random_state=seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

# Modelo Keras simple
def build_binary_model():
    model = keras.models.Sequential([
        keras.layers.Input(shape=(2,)),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dense(8, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_binary_model()
history = model.fit(X_train_s, y_train, validation_data=(X_test_s, y_test), epochs=100, batch_size=32, verbose=0)

# Evaluación
train_loss, train_acc = model.evaluate(X_train_s, y_train, verbose=0)
test_loss, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
print(f'Train acc: {train_acc:.4f} — Test acc: {test_acc:.4f}')

# Curvas de entrenamiento
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title('Loss')
plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend()
plt.title('Accuracy')
plt.show()

# Mapa de decisión (visualización rápida)
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 300),
                     np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
grid_s = scaler.transform(grid)
probs = model.predict(grid_s, verbose=0).reshape(xx.shape)
plt.figure(figsize=(6,5))
plt.contourf(xx, yy, probs, levels=50, cmap='RdBu', alpha=0.6)
plt.scatter(X_test[:,0], X_test[:,1], c=y_test, cmap='bwr', edgecolor='k', s=30)
plt.title('Decision boundary (test set)')
plt.show()

## Ejercicio 2 — `load_digits` (clasificación multiclase, 6 clases)

In [ ]:
# Cargar datos
digits = load_digits(n_class=6)  # seguir el ejemplo original
X, y = digits.data, digits.target
n_samples, n_features = X.shape
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

# Codificar etiquetas para loss categorical_crossentropy
# Compatibilidad con diferentes versiones de scikit-learn (sparse -> sparse_output)
try:
    encoder = OneHotEncoder(sparse_output=False, categories='auto')
except TypeError:
    encoder = OneHotEncoder(sparse=False, categories='auto')
y_train_o = encoder.fit_transform(y_train.reshape(-1,1))
y_test_o = encoder.transform(y_test.reshape(-1,1))
n_classes = y_train_o.shape[1]

# Modelo Keras para multiclasificación
def build_multiclass_model(input_dim, n_classes):
    model = keras.models.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(n_classes, activation='softmax')
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model_m = build_multiclass_model(n_features, n_classes)
history_m = model_m.fit(X_train_s, y_train_o, validation_data=(X_test_s, y_test_o), epochs=80, batch_size=32, verbose=0)

# Evaluación
train_loss, train_acc = model_m.evaluate(X_train_s, y_train_o, verbose=0)
test_loss, test_acc = model_m.evaluate(X_test_s, y_test_o, verbose=0)
print(f'Train acc: {train_acc:.4f} — Test acc: {test_acc:.4f}')

# Predicciones y reporte
y_pred_proba = model_m.predict(X_test_s, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
print(classification_report(y_test, y_pred, digits=4))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Pred')
plt.ylabel('True')
plt.title('Confusion matrix (test)')
plt.show()